# Lecture 22: Interactions, Mixed Effects, and Logistic Regression
**Ling 250/450: Data Science for Linguistics** | Spring 2026

## Overview

This notebook extends the linear regression framework from Lecture 21 in three directions:

1. **Interaction terms**: allowing the effect of one predictor to vary depending on another
2. **Mixed-effects models**: handling grouped data (e.g., multiple observations per speaker) without violating the independence assumption
3. **Logistic regression**: modeling binary outcomes (yes/no, accepted/rejected) instead of continuous ones

In [ ]:
# Ensure UTF-8 locale so non-ASCII characters (e.g. IPA) display correctly
Sys.setlocale("LC_ALL", "en_US.UTF-8")

# lme4 provides lmer() for mixed-effects models
# lmerTest extends lme4 to report p-values in summaries
# If not installed: install.packages(c("lme4", "lmerTest"))
library(lme4)
library(lmerTest)

---

## Part 1: Interaction Terms

In the additive model from Lecture 21:

$$y = \alpha + \beta_1 x_1 + \beta_2 x_2 + \epsilon$$

the effect of $x_1$ on $y$ is always $\beta_1$, regardless of $x_2$. The two predictors contribute **independently**.

An **interaction** breaks that assumption. It says: *the effect of $x_1$ depends on the value of $x_2$.*

In R, an interaction term is added with `:` (the interaction alone) or `*` (main effects plus interaction):

```r
lm(y ~ x1 + x2 + x1:x2)   # main effects + interaction
lm(y ~ x1 * x2)            # shorthand for the same thing
```

The model becomes:

$$y = \alpha + \beta_1 x_1 + \beta_2 x_2 + \beta_{12} (x_1 \times x_2) + \epsilon$$

The interaction coefficient $\beta_{12}$ tells you **how much the slope of $x_1$ changes** for each one-unit increase in $x_2$.

### Visualizing the Difference

The easiest way to see whether an interaction is present is to plot the data. In the algae/pond example:

- **No interaction**: both ponds respond to fertilizer at the same rate — the regression lines are **parallel**
- **Interaction**: the ponds respond at different rates — the lines **diverge** (or cross)

Let's generate two datasets — one with no interaction, one with a strong interaction — to see this visually.

In [ ]:
# Dataset A: additive (no interaction) — same as Lecture 21
set.seed(1)
n_per_pond <- 12
pond_additive <- data.frame(
  fertilizer = round(runif(2 * n_per_pond, min = 0, max = 10), 2),
  pond = factor(c(rep(1, n_per_pond), rep(2, n_per_pond)))
)
pond_additive$algae <- round(
  2.0 * pond_additive$fertilizer +
  3.5 * (pond_additive$pond == 2) +
  rnorm(2 * n_per_pond, mean = 0, sd = 1),
  2
)

# Dataset B: interaction — pond 2 responds much more strongly to fertilizer
set.seed(1)
pond_interaction <- data.frame(
  fertilizer = round(runif(2 * n_per_pond, min = 0, max = 10), 2),
  pond = factor(c(rep(1, n_per_pond), rep(2, n_per_pond)))
)
pond_interaction$algae <- round(
  0.8 * pond_interaction$fertilizer +                                # pond 1 slope
  2.0 * (pond_interaction$pond == 2) +                               # intercept shift
  2.5 * (pond_interaction$pond == 2) * pond_interaction$fertilizer + # interaction: extra slope for pond 2
  3.0 + rnorm(2 * n_per_pond, mean = 0, sd = 1.5),
  2
)

In [ ]:
pond_colors <- c("1" = "steelblue", "2" = "salmon")

par(mfrow = c(1, 2))

# --- Left: no interaction (parallel lines) ---
plot(
  pond_additive$fertilizer, pond_additive$algae,
  col  = pond_colors[as.character(pond_additive$pond)],
  pch  = 19, xlab = "Fertilizer", ylab = "Algae",
  main = "No Interaction\n(parallel lines)"
)
mod_add <- lm(algae ~ fertilizer + pond, data = pond_additive)
coefs_add <- coef(mod_add)
abline(a = coefs_add["(Intercept)"],            b = coefs_add["fertilizer"], col = "steelblue", lwd = 2)
abline(a = coefs_add["(Intercept)"] + coefs_add["pond2"], b = coefs_add["fertilizer"], col = "salmon",    lwd = 2)
legend("topleft", legend = c("Pond 1", "Pond 2"), col = c("steelblue", "salmon"), pch = 19, lty = 1)

# --- Right: interaction (diverging lines) ---
plot(
  pond_interaction$fertilizer, pond_interaction$algae,
  col  = pond_colors[as.character(pond_interaction$pond)],
  pch  = 19, xlab = "Fertilizer", ylab = "Algae",
  main = "With Interaction\n(diverging lines)"
)
mod_int <- lm(algae ~ fertilizer * pond, data = pond_interaction)
coefs_int <- coef(mod_int)
abline(a = coefs_int["(Intercept)"],                         b = coefs_int["fertilizer"],                           col = "steelblue", lwd = 2)
abline(a = coefs_int["(Intercept)"] + coefs_int["pond2"],   b = coefs_int["fertilizer"] + coefs_int["fertilizer:pond2"], col = "salmon",    lwd = 2)
legend("topleft", legend = c("Pond 1", "Pond 2"), col = c("steelblue", "salmon"), pch = 19, lty = 1)

par(mfrow = c(1, 1))

The parallel lines on the left mean the fertilizer effect is identical for both ponds — pond 2 just starts higher. The diverging lines on the right mean pond 2 *responds more strongly* to fertilizer: at low fertilizer levels the ponds are similar, but at high levels they diverge dramatically.

### Fitting and Interpreting an Interaction Model

In [ ]:
# Additive model (no interaction)
mod_additive <- lm(algae ~ fertilizer + pond, data = pond_interaction)
summary(mod_additive)

In [ ]:
# Interaction model
mod_interaction <- lm(algae ~ fertilizer * pond, data = pond_interaction)
summary(mod_interaction)

### Interpreting the Coefficients

The interaction model adds a new row to the coefficient table: `fertilizer:pond2`.

| Coefficient | Meaning |
|-------------|--------|
| `(Intercept)` | Predicted algae for **pond 1** when fertilizer = 0 |
| `fertilizer` | Slope for **pond 1** — how much algae increases per unit of fertilizer |
| `pond2` | Intercept difference between pond 2 and pond 1 (at fertilizer = 0) |
| `fertilizer:pond2` | **Extra slope for pond 2** — how much steeper pond 2's response is than pond 1's |

So the slopes are:
- Pond 1 slope: $\hat{\beta}_{\text{fertilizer}}$
- Pond 2 slope: $\hat{\beta}_{\text{fertilizer}} + \hat{\beta}_{\text{fertilizer:pond2}}$

If `fertilizer:pond2` is positive, pond 2 responds more strongly. If it's negative, pond 2 responds less strongly.

**Checking:** The true model had pond 1 slope = 0.8 and interaction = 2.5, so the true pond 2 slope is 3.3. How close did `lm()` get?

> **Rule of thumb**: if the interaction term isn't significant, an additive model is usually preferred (simpler, easier to interpret). Only add interactions when there is a theoretical reason or visual evidence for them.

<details>
<summary><b>Challenge:</b> Using the clinical trial data from Lecture 21, fit a model with the interaction between <code>drug</code> and <code>therapy</code>. Is the interaction significant? What does it mean if it is?</summary>

```r
clin_trial <- data.frame(
  drug = factor(
    c(rep("placebo", 6), rep("anxifree", 6), rep("joyzepam", 6)),
    levels = c("placebo", "anxifree", "joyzepam")
  ),
  therapy = factor(rep(c(rep("no.therapy", 3), rep("CBT", 3)), 3)),
  mood.gain = c(
    0.5, 0.3, 0.1, 0.6, 0.9, 0.3,
    0.6, 0.4, 0.2, 1.1, 0.8, 1.2,
    1.4, 1.7, 1.3, 1.8, 1.3, 1.4
  )
)

mod_clin <- lm(mood.gain ~ drug * therapy, data = clin_trial)
summary(mod_clin)
```

A significant `drug:therapyCBT` interaction would mean CBT amplifies (or dampens) the drug effect differently for each drug — e.g., that joyzepam plus CBT produces *more* improvement than you'd predict from adding the drug effect and the CBT effect separately.
</details>

---

## Part 2: Mixed-Effects Models

### The Independence Problem

All the models so far assume **independent observations** — each data point tells us something new, unrelated to the others.

But linguistic data almost never works this way:

- A vowel study collects 30 tokens from each of 20 speakers → 600 data points, but they're not all independent
- A reaction-time experiment runs 50 trials per participant → observations within a participant are correlated
- A corpus study samples multiple sentences per text → texts from the same author share style

Ignoring this structure leads to **pseudoreplication**: the model thinks it has more independent evidence than it really does, producing overconfident (anti-conservative) p-values.

### What Random Effects Do

A **random effect** models the variation *between groups* (speakers, items, texts) as coming from a probability distribution, rather than estimating a separate coefficient for each group.

Instead of:
$$y_{ij} = \alpha + \beta x_{ij} + \gamma_{j} + \epsilon_{ij} \quad \text{(fixed effect per group } j \text{)}$$

we write:
$$y_{ij} = \alpha + \beta x_{ij} + b_j + \epsilon_{ij} \quad \text{where } b_j \sim \mathcal{N}(0, \sigma^2_b)$$

The $b_j$ values are **random intercepts**: each group's baseline is allowed to deviate from the population mean $\alpha$, but those deviations are assumed to be normally distributed. The model estimates $\sigma^2_b$ (how variable the groups are) rather than each $b_j$ individually.

A model with both fixed effects (the predictors we care about) and random effects (the grouping structure we're controlling for) is called a **mixed-effects model**.

### Four Ways to Model Grouped Data

The figure below shows the full range of options, from a single fixed regression line to a model with group-specific intercepts *and* slopes.

In [ ]:
set.seed(1)

col_red  <- "#C45252"
col_dark <- "#2E4080"
col_teal <- "#2A8870"
group_cols <- c(col_red, col_dark, col_teal)

# Helper: scatter n points around the line y = a + b*x
line_pts <- function(n, a, b, xmin = 1, xmax = 9, sigma = 0.55) {
  xs <- runif(n, xmin, xmax)
  ys <- a + b * xs + rnorm(n, 0, sigma)
  list(x = xs, y = ys)
}

par(mfrow = c(2, 2), mar = c(2, 3.8, 3.5, 1.5))

# === Panel 1: Fixed Effects / Simple Linear Regression ===
pts1 <- line_pts(22, a = 10, b = -0.9, sigma = 1.0)
plot(
  pts1$x, pts1$y, pch = 19, cex = 0.85, col = col_dark,
  xaxt = "n", yaxt = "n", xlab = "", ylab = "",
  xlim = c(0, 10), ylim = c(0, 16),
  main = "Fixed Effects\nSimple Linear Regression"
)
abline(a = 10, b = -0.9, col = "gray30", lwd = 2.5)
axis(2, at = 10, labels = expression(alpha), las = 1, tick = FALSE, cex.axis = 1.15)
text(8.6, 10 - 0.9 * 8.6 - 1.0, expression(beta), cex = 1.15, col = "gray20")

# === Panel 2: Mixed Effects / Random Intercept, Fixed Slope ===
int_ri     <- c(13, 10, 7)  # alpha_1 > alpha_2 > alpha_3
slope_ri   <- -0.9
pts2 <- lapply(1:3, function(i) line_pts(8, int_ri[i], slope_ri))
plot(
  NULL, xaxt = "n", yaxt = "n", xlab = "", ylab = "",
  xlim = c(0, 10), ylim = c(0, 16),
  main = "Mixed Effects\nRandom Intercept, Fixed Slope"
)
for (i in 1:3) {
  points(pts2[[i]]$x, pts2[[i]]$y, pch = 19, cex = 0.85, col = group_cols[i])
  abline(a = int_ri[i], b = slope_ri, col = group_cols[i], lwd = 2.5)
}
axis(2, at = int_ri,
     labels = c(expression(alpha[1]), expression(alpha[2]), expression(alpha[3])),
     las = 1, tick = FALSE, cex.axis = 1.0)
text(8.7, int_ri[2] + slope_ri * 8.7 - 0.9, expression(beta), cex = 1.15, col = "gray20")

# === Panel 3: Mixed Effects / Fixed Intercept, Random Slope ===
int_fi   <- 10
slopes_rs <- c(-0.15, -0.9, -1.8)  # beta_1 (flat/red), beta_2 (medium/dark), beta_3 (steep/teal)
pts3 <- lapply(1:3, function(i) line_pts(8, int_fi, slopes_rs[i]))
plot(
  NULL, xaxt = "n", yaxt = "n", xlab = "", ylab = "",
  xlim = c(0, 10), ylim = c(-8, 14),
  main = "Mixed Effects\nFixed Intercept, Random Slope"
)
for (i in 1:3) {
  points(pts3[[i]]$x, pts3[[i]]$y, pch = 19, cex = 0.85, col = group_cols[i])
  abline(a = int_fi, b = slopes_rs[i], col = group_cols[i], lwd = 2.5)
}
axis(2, at = int_fi, labels = expression(alpha), las = 1, tick = FALSE, cex.axis = 1.15)
for (i in 1:3) {
  ypos   <- int_fi + slopes_rs[i] * 9.2
  offset <- c(0.9, 0.8, -0.8)[i]
  text(9.5, ypos + offset, bquote(beta[.(i)]), cex = 1.0, col = group_cols[i])
}

# === Panel 4: Random Effects / Random Intercept, Random Slope ===
# Red (i=1): alpha_1 = 11, moderate slope -> label beta_1
# Dark (i=2): alpha_2 = 7,  near-flat slope   -> label beta_3
# Teal (i=3): alpha_3 = 14, steep slope        -> label beta_2
int_full   <- c(11,   7,   14)
slope_full <- c(-0.75, -0.08, -1.7)
beta_labs  <- list(expression(beta[1]), expression(beta[3]), expression(beta[2]))
pts4 <- lapply(1:3, function(i) line_pts(8, int_full[i], slope_full[i]))
plot(
  NULL, xaxt = "n", yaxt = "n", xlab = "", ylab = "",
  xlim = c(0, 10), ylim = c(-5, 18),
  main = "Random Effects\nRandom Intercept, Random Slope"
)
for (i in 1:3) {
  points(pts4[[i]]$x, pts4[[i]]$y, pch = 19, cex = 0.85, col = group_cols[i])
  abline(a = int_full[i], b = slope_full[i], col = group_cols[i], lwd = 2.5)
}
# Label intercepts on y-axis (teal = alpha_3 highest, red = alpha_1 middle, dark = alpha_2 lowest)
axis(2, at = c(int_full[3], int_full[1], int_full[2]),
     labels = c(expression(alpha[3]), expression(alpha[1]), expression(alpha[2])),
     las = 1, tick = FALSE, cex.axis = 1.0)
for (i in 1:3) {
  ypos   <- int_full[i] + slope_full[i] * 9.2
  offset <- c(0.8, 0.8, -0.8)[i]
  text(9.5, ypos + offset, beta_labs[[i]], cex = 1.0, col = group_cols[i])
}

par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1)

**Reading the figure:**

| Panel | What varies across groups | What's shared |
|-------|--------------------------|---------------|
| Top-left | nothing — one group | one intercept $\alpha$, one slope $\beta$ |
| Top-right | intercepts $\alpha_1, \alpha_2, \alpha_3$ | slope $\beta$ |
| Bottom-left | slopes $\beta_1, \beta_2, \beta_3$ | intercept $\alpha$ |
| Bottom-right | intercepts *and* slopes | only the population-level distributions |

In linguistics, **random intercepts by speaker** (top-right) are the most common. Each speaker has their own baseline, but the effect of the predictor is assumed to be the same for everyone.

**Random slopes by speaker** (bottom panels) are more flexible: the effect of the predictor can differ across speakers. They require more data (you need enough observations per speaker to estimate a slope) and are often harder to fit.

### R Syntax: `lmer()`

Mixed-effects models in R use the `lmer()` function from the `lme4` package:

```r
# Random intercept only
lmer(y ~ x + (1 | group), data = df)

# Random intercept and random slope for x
lmer(y ~ x + (1 + x | group), data = df)

# Random slope only (uncommon; rarely theoretically motivated)
lmer(y ~ x + (0 + x | group), data = df)
```

The **random effects specification** `(... | group)` is the key new syntax:
- The left side of `|` describes what varies by group: `1` = intercept, `x` = slope for `x`
- The right side names the grouping variable

Fixed effects go on the left of `~`, just like in `lm()`.

### Example: Vowel Formants and Speaker Variation

The vowel dataset has F1 and F2 measurements for multiple vowels from multiple speakers. We want to know whether F2 predicts F1 — but each speaker has their own overall vowel space (some speakers have higher formants in general, some lower). We need to control for that.

Without a random effect, observations from the same speaker are treated as independent, which they're not. A **random intercept for speaker** accounts for the fact that each speaker has their own baseline.

In [ ]:
raw_lines <- readLines("../data/vowels.csv", encoding = "UTF-8", warn = FALSE)
vowels <- read.csv(textConnection(paste(raw_lines, collapse = "
")))
head(vowels)

In [ ]:
# How many observations per speaker?
table(vowels$SPEAKER)

In [ ]:
# Fixed-effects only model (ignores speaker structure)
mod_fixed <- lm(F1 ~ F2, data = vowels)

# Mixed-effects model with random intercept for speaker
mod_ri <- lmer(F1 ~ F2 + (1 | SPEAKER), data = vowels)

summary(mod_ri)

### Reading the `lmer()` Summary

The output has two sections:

**Random effects** — variation due to the grouping structure:
- `(Intercept)` under `SPEAKER`: the standard deviation of speaker-level intercepts (how different speakers' baselines are)
- `Residual`: standard deviation of observation-level error (leftover unexplained variation)

**Fixed effects** — the coefficients, just like `lm()`:
- `(Intercept)`: population-level baseline F1 (average across speakers, at F2 = 0)
- `F2`: effect of F2 on F1, controlling for speaker differences

The p-values from `lmerTest` use the Satterthwaite approximation for degrees of freedom (a technically complex issue — the key takeaway is that they're valid, just approximate).

In [ ]:
# Visualize: all data in gray, per-speaker random intercept lines in color
speakers     <- unique(vowels$SPEAKER)
n_speakers   <- length(speakers)
speaker_cols <- rainbow(n_speakers, s = 0.7, v = 0.85)

plot(
  vowels$F2, vowels$F1,
  col  = adjustcolor("gray60", alpha.f = 0.4),
  pch  = 19, cex = 0.6,
  xlab = "F2 (Hz)", ylab = "F1 (Hz)",
  main = "F1 ~ F2: population line + per-speaker random intercepts"
)

# Population-level (fixed effects) line
abline(
  a = fixef(mod_ri)["(Intercept)"],
  b = fixef(mod_ri)["F2"],
  col = "black", lwd = 3, lty = 2
)

# Each speaker's line: fixed slope, shifted intercept
ranef_intercepts <- ranef(mod_ri)$SPEAKER[, "(Intercept)"]
for (i in seq_along(speakers)) {
  speaker_int <- fixef(mod_ri)["(Intercept)"] + ranef_intercepts[i]
  abline(
    a   = speaker_int,
    b   = fixef(mod_ri)["F2"],
    col = adjustcolor(speaker_cols[i], alpha.f = 0.75),
    lwd = 1.5
  )
}

legend("topright",
  legend = c("Population line", "Per-speaker lines"),
  col    = c("black", speaker_cols[1]),
  lty    = c(2, 1), lwd = c(3, 1.5)
)

All per-speaker lines are parallel (same slope, different intercepts) — that's the defining feature of a random-intercept-only model.

Compare this to the population line (dashed): it has the same slope as all the per-speaker lines, but its intercept is the average across all speakers.

### Adding a Random Slope

The random-intercept model assumes F2 has the same effect on F1 for every speaker. But speakers could differ in how strongly F1 and F2 co-vary in their vowel space. A **random slope** for F2 allows that effect to vary by speaker.

The formula changes from `(1 | SPEAKER)` to `(1 + F2 | SPEAKER)`:

In [ ]:
mod_rs <- lmer(F1 ~ F2 + (1 + F2 | SPEAKER), data = vowels)
summary(mod_rs)

In [ ]:
# Per-speaker lines now have different slopes AND different intercepts
plot(
  vowels$F2, vowels$F1,
  col  = adjustcolor("gray60", alpha.f = 0.4),
  pch  = 19, cex = 0.6,
  xlab = "F2 (Hz)", ylab = "F1 (Hz)",
  main = "F1 ~ F2: random intercepts + random slopes by speaker"
)
abline(
  a = fixef(mod_rs)["(Intercept)"],
  b = fixef(mod_rs)["F2"],
  col = "black", lwd = 3, lty = 2
)

ranef_spkr <- ranef(mod_rs)$SPEAKER
for (i in seq_along(speakers)) {
  speaker_int   <- fixef(mod_rs)["(Intercept)"] + ranef_spkr[i, "(Intercept)"]
  speaker_slope <- fixef(mod_rs)["F2"]           + ranef_spkr[i, "F2"]
  abline(
    a   = speaker_int,
    b   = speaker_slope,
    col = adjustcolor(speaker_cols[i], alpha.f = 0.75),
    lwd = 1.5
  )
}
legend("topright",
  legend = c("Population line", "Per-speaker lines"),
  col    = c("black", speaker_cols[1]),
  lty    = c(2, 1), lwd = c(3, 1.5)
)

Now the per-speaker lines are **no longer parallel** — they can cross, converge, or diverge. Whether the added complexity is warranted depends on whether the random slope variance is non-trivial (check the `Std.Dev.` for the `F2` row under Random effects).

> **Model comparison note**: to formally test whether random slopes improve fit, you can use `anova(mod_ri, mod_rs)`. But for this course, visual inspection and theoretical motivation are usually enough to guide the choice.

<details>
<summary><b>Challenge:</b> Fit a model predicting F1 from <code>SEX</code> (a fixed effect) with a random intercept for <code>SPEAKER</code>. Is there a significant effect of speaker sex on F1? Does this make phonetic sense?</summary>

```r
mod_sex <- lmer(F1 ~ SEX + (1 | SPEAKER), data = vowels)
summary(mod_sex)
```

Expected result: female speakers tend to have higher F1 than male speakers on average (due to differences in vocal tract size), so `SEXmale` should have a negative coefficient. Note, however, that with speaker as a random effect, SEX is a between-speaker predictor — its estimate reflects variation *across* speakers, not within them.
</details>

---

## Part 3: Logistic Regression

### Binary Outcomes in Linguistics

So far, all our regression models have predicted a **continuous** outcome (algae growth, F1 in Hz, mood gain). But many linguistic questions have **binary** outcomes:

- Did the listener hear /p/ or /b/? (phoneme categorization)
- Did the speaker use the irregular past tense form, or the regularized one?
- Was the sentence judged grammatical or ungrammatical?
- Does the word appear in reduced form ("gonna") or full form ("going to")?

**Logistic regression** is the standard tool for these cases. Instead of predicting the outcome directly, it predicts the **probability** that the outcome is 1 (e.g., "heard as /p/").

### Why Not Just Use Linear Regression?

Let's see what happens if we try to fit a linear regression to a binary outcome.

In [ ]:
# Simulate a VOT perception experiment.
# Participants heard stimuli with varying Voice Onset Time (VOT, in ms)
# and categorized each token as /b/ (0) or /p/ (1).
# Short VOT -> /b/;  long VOT -> /p/;  boundary around 35 ms.
set.seed(1)
n_trials <- 120
vot_ms   <- runif(n_trials, min = 0, max = 80)

# True logistic model: log-odds = -6.0 + 0.17 * VOT
true_logit <- -6.0 + 0.17 * vot_ms
prob_p     <- 1 / (1 + exp(-true_logit))
heard_p    <- rbinom(n_trials, size = 1, prob = prob_p)

vot_data <- data.frame(vot = vot_ms, heard_p = heard_p)
head(vot_data)

In [ ]:
# Fit a linear regression to binary data — notice the problems
mod_linear <- lm(heard_p ~ vot, data = vot_data)

vot_seq <- seq(0, 80, length.out = 200)
pred_linear <- predict(mod_linear, newdata = data.frame(vot = vot_seq))

plot(
  vot_data$vot, vot_data$heard_p,
  pch = 19, cex = 0.7, col = adjustcolor("#334477", alpha.f = 0.5),
  xlab = "Voice Onset Time (ms)", ylab = "Heard as /p/ (1 = yes)",
  main = "Binary outcome — linear regression fit",
  ylim = c(-0.2, 1.2)
)
lines(vot_seq, pred_linear, col = "firebrick", lwd = 2)
abline(h = c(0, 1), lty = 2, col = "gray60")
text(5,  -0.15, "/b/", cex = 0.9, col = "gray40")
text(75,  1.15, "/p/", cex = 0.9, col = "gray40")

Two problems with linear regression here:

1. **Predicted values go outside [0, 1]** — a probability below 0 or above 1 is meaningless
2. **The shape is wrong** — the true pattern is S-shaped (slow change at extremes, fast change near the boundary), not linear

Logistic regression solves both problems by modeling the **log-odds** of the outcome, then transforming back to a probability using the **logistic function**:

$$P(y = 1) = \frac{1}{1 + e^{-(\alpha + \beta x)}} = \sigma(\alpha + \beta x)$$

The logistic function $\sigma(z)$ maps any real number to $(0, 1)$, producing the characteristic S-curve.

In [ ]:
# Illustrate the logistic function shape
z <- seq(-6, 6, length.out = 200)
sigma_z <- 1 / (1 + exp(-z))

plot(
  z, sigma_z,
  type = "l", lwd = 2.5, col = "#2E4080",
  xlab = expression(z == alpha + beta * x),
  ylab = expression(sigma(z)),
  main = expression("Logistic function: " * sigma(z) == frac(1, 1 + e^{-z})),
  ylim = c(0, 1)
)
abline(h = 0.5, lty = 2, col = "gray60")
abline(v = 0,   lty = 2, col = "gray60")
text(-5.5, 0.92, "P(y=1) → 1 as z → ∞",  cex = 0.85, col = "gray40")
text( 5.5, 0.08, "P(y=1) → 0 as z → -∞", cex = 0.85, col = "gray40", adj = 1)

### Fitting with `glm()`

Logistic regression is a special case of **generalized linear models** (GLMs). In R, it uses `glm()` with `family = binomial`:

In [ ]:
mod_logistic <- glm(heard_p ~ vot, data = vot_data, family = binomial)
summary(mod_logistic)

In [ ]:
# Plot the data and the fitted logistic curve
pred_logistic <- predict(mod_logistic,
  newdata = data.frame(vot = vot_seq),
  type = "response"   # returns P(y=1), not log-odds
)

plot(
  vot_data$vot, vot_data$heard_p,
  pch = 19, cex = 0.7, col = adjustcolor("#334477", alpha.f = 0.5),
  xlab = "Voice Onset Time (ms)", ylab = "P(heard as /p/)",
  main = "VOT perception: logistic regression fit",
  ylim = c(-0.05, 1.05)
)
lines(vot_seq, pred_logistic, col = "firebrick", lwd = 2.5)
abline(h = 0.5, lty = 2, col = "gray60")

# Mark the estimated perceptual boundary (where P = 0.5, i.e. log-odds = 0)
boundary <- -coef(mod_logistic)["(Intercept)"] / coef(mod_logistic)["vot"]
abline(v = boundary, lty = 3, col = "gray40")
text(boundary + 1, 0.05, sprintf("boundary\n%.0f ms", boundary), cex = 0.85, adj = 0, col = "gray40")

### Interpreting the Coefficients

The `summary()` output looks similar to `lm()`, but the coefficients are now in **log-odds**, not in the units of $y$:

$$\log\left(\frac{P(y=1)}{P(y=0)}\right) = \alpha + \beta x$$

| Coefficient | Meaning |
|-------------|--------|
| `(Intercept)` $\alpha$ | Log-odds of hearing /p/ when VOT = 0 ms |
| `vot` $\beta$ | Change in log-odds of /p/ per 1 ms increase in VOT |

Log-odds are hard to interpret directly. Two conversions:

**Odds ratios**: $e^{\hat{\beta}}$ — how much the *odds* of the outcome multiply for each 1-unit increase in $x$. If $e^{\hat{\beta}} = 1.18$, the odds increase by 18% per ms of VOT.

**Perceptual boundary**: the VOT at which $P(y=1) = 0.5$ — i.e., where the listener is at chance. This occurs when $\alpha + \beta x = 0$, giving $x^* = -\alpha/\beta$.

In [ ]:
# Odds ratio for VOT
odds_ratio <- exp(coef(mod_logistic)["vot"])
cat(sprintf(
  "Odds ratio for VOT: %.3f\n(each additional ms of VOT multiplies the odds of /p/ by %.3f)\n",
  odds_ratio, odds_ratio
))

# Estimated perceptual boundary
cat(sprintf(
  "Estimated perceptual boundary: %.1f ms\nTrue boundary: 35.3 ms\n",
  boundary
))

### Significance Testing in `glm()`

The `summary()` shows **Wald z-tests** (not t-tests) for each coefficient — functionally similar to what `lm()` reports, but based on normal-distribution approximations. The p-values have the same interpretation.

For comparing nested models (e.g., does adding a second predictor improve fit?), use `anova(mod1, mod2, test = "Chisq")` — a likelihood ratio test.

<details>
<summary><b>Challenge:</b> Add a second predictor <code>speaking_rate</code> to the VOT model. Simulate <code>speaking_rate</code> as a random variable (e.g., <code>runif(n_trials, 3, 8)</code> for syllables/second). Add it to <code>vot_data</code> and fit a new logistic model with both <code>vot</code> and <code>speaking_rate</code>. Is speaking rate a significant predictor once VOT is accounted for?</summary>

```r
set.seed(1)  # reset seed to match the original data
vot_data$speaking_rate <- runif(n_trials, 3, 8)

mod_two_pred <- glm(
  heard_p ~ vot + speaking_rate,
  data = vot_data,
  family = binomial
)
summary(mod_two_pred)
```

Since `speaking_rate` was generated independently of the outcome in this simulated dataset, it should *not* be significant — a good sanity check. In real data, speaking rate genuinely affects phoneme categorization (faster speech → shifted category boundaries).
</details>

---

## Quick Reference

### Interaction Terms

| Task | R code |
|------|--------|
| Additive model (no interaction) | `lm(y ~ x1 + x2, data = df)` |
| Model with interaction | `lm(y ~ x1 * x2, data = df)` |
| Interaction term only (no main effect) | `lm(y ~ x1 + x2 + x1:x2, data = df)` |

**Reading interaction coefficients**: `x1:x2group2` = how much steeper (or shallower) the slope of `x1` is for `group2` compared to the baseline group.

### Mixed-Effects Models (`lme4`)

| Task | R code |
|------|--------|
| Random intercept by group | `lmer(y ~ x + (1 \| group), data = df)` |
| Random intercept + slope | `lmer(y ~ x + (1 + x \| group), data = df)` |
| Get fixed-effect coefficients | `fixef(model)` |
| Get random-effect deviations | `ranef(model)` |
| Get predicted values | `predict(model)` |

### Logistic Regression

| Task | R code |
|------|--------|
| Fit logistic model | `glm(y ~ x, data = df, family = binomial)` |
| Predicted probabilities | `predict(model, newdata = df, type = "response")` |
| Predicted log-odds | `predict(model, newdata = df, type = "link")` |
| Odds ratios | `exp(coef(model))` |
| Compare nested models | `anova(mod1, mod2, test = "Chisq")` |

### Key Terms

| Term | Meaning |
|------|---------|
| Interaction | The effect of one predictor depends on the value of another |
| Random effect | Variation attributed to a grouping variable (speaker, item, text) |
| Random intercept | Each group gets its own baseline, drawn from $\mathcal{N}(0, \sigma^2_b)$ |
| Random slope | Each group gets its own slope for a predictor |
| Fixed effect | The predictor coefficients we care about inferring |
| Log-odds (logit) | $\log[P/(1-P)]$ — the linear predictor in logistic regression |
| Odds ratio | $e^{\hat{\beta}}$ — multiplicative change in odds per unit of $x$ |
| Perceptual boundary | VOT (or similar) at which $P(\text{response}) = 0.5$ |